In [1]:
%pip install lightgbm wandb -Uq

import os
import getpass

import wandb

_wandb_api_key_env = os.environ.get("WANDB_API_KEY")
os.environ.pop("WANDB_API_KEY", None)
try:
    wandb.logout()
except Exception:
    pass

WANDB_ENTITY = "toberi23-free-university-of-tbilisi-"
WANDB_PROJECT = "Walmart-Recruiting-Store-Sales-Forecasting"

wandb_key = _wandb_api_key_env or getpass.getpass("Paste your W&B API key: ").strip()
wandb.login(key=wandb_key, relogin=True, force=True, verify=True)

who = wandb.Api().viewer
print("Logged in as:", who.username, "| entity:", who.entity)


[notice] A new release of pip is available: 25.2 -> 26.1.2
[notice] To update, run: pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.


wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.


wandb: [wandb.login()] Using explicit session credentials for https://api.wandb.ai.


wandb: Appending key for api.wandb.ai to your netrc file: /Users/lizamarsagishvili/.netrc


wandb: Currently logged in as: lmars23 (toberi23-free-university-of-tbilisi-) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


Logged in as: lmars23 | entity: toberi23-free-university-of-tbilisi-


In [2]:
import os, time, pickle, warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd

from sklearn.base import BaseEstimator, TransformerMixin, RegressorMixin

import lightgbm as lgb

import wandb

SEED = 42
np.random.seed(SEED)

WANDB_GROUP = "LightGBM_Training"
pd.set_option("display.width", 200)

In [3]:
_LOCAL_DATA_DIR = os.path.join("data", "walmart-recruiting-store-sales-forecasting")
_KAGGLE_DATA_DIR = "/kaggle/input/competitions/walmart-recruiting-store-sales-forecasting"

COMP = (
    os.environ.get("WALMART_DATA_DIR")
    or (_LOCAL_DATA_DIR if os.path.isdir(_LOCAL_DATA_DIR) else _KAGGLE_DATA_DIR)
)
print("Reading competition data from:", COMP)

def load_merged(kind: str = "train") -> pd.DataFrame:
    if kind not in ("train", "test"):
        raise ValueError("kind must be 'train' or 'test'")
    base = pd.read_csv(f"{COMP}/{kind}.csv.zip")
    base["Date"] = pd.to_datetime(base["Date"])
    stores = pd.read_csv(f"{COMP}/stores.csv")
    feats = pd.read_csv(f"{COMP}/features.csv.zip")
    feats["Date"] = pd.to_datetime(feats["Date"])
    feats = feats.drop(columns=["IsHoliday"])
    return (
        base.merge(stores, on="Store", how="left")
            .merge(feats, on=["Store", "Date"], how="left")
            .sort_values(["Store", "Dept", "Date"])
            .reset_index(drop=True)
    )

df_train = load_merged("train")
df_test = load_merged("test")
print("Train:", df_train.shape, "| Test:", df_test.shape)
df_train.head()

Reading competition data from: data/walmart-recruiting-store-sales-forecasting


Train: (421570, 16) | Test: (115064, 15)


,Store,Dept,Date,Weekly_Sales,IsHoliday,Type,Size,Temperature,Fuel_Price,MarkDown1,MarkDown2,MarkDown3,MarkDown4,MarkDown5,CPI,Unemployment
0,1,1,2010-02-05,24924.50,False,A,151315,42.31,2.572,NaN,NaN,NaN,NaN,NaN,211.096358,8.106
1,1,1,2010-02-12,46039.49,True,A,151315,38.51,2.548,NaN,NaN,NaN,NaN,NaN,211.242170,8.106
2,1,1,2010-02-19,41595.55,False,A,151315,39.93,2.514,NaN,NaN,NaN,NaN,NaN,211.289143,8.106
3,1,1,2010-02-26,19403.54,False,A,151315,46.63,2.561,NaN,NaN,NaN,NaN,NaN,211.319643,8.106
4,1,1,2010-03-05,21827.90,False,A,151315,46.50,2.625,NaN,NaN,NaN,NaN,NaN,211.350143,8.106


In [4]:
def wmae(y_true, y_pred, is_holiday):
    w = np.where(np.asarray(is_holiday, dtype=bool), 5.0, 1.0)
    return float(np.sum(w * np.abs(np.asarray(y_true) - np.asarray(y_pred))) / np.sum(w))

assert abs(wmae([10, 20], [8, 15], [True, False]) - 15 / 6) < 1e-9

VAL_START = pd.Timestamp("2011-11-04")
VAL_END = pd.Timestamp("2012-07-27")

cv_train = df_train[df_train.Date < VAL_START].copy()
cv_val = df_train[(df_train.Date >= VAL_START) & (df_train.Date <= VAL_END)].copy()

print("CV train:", cv_train.shape, cv_train.Date.min().date(), "->", cv_train.Date.max().date())
print("CV val  :", cv_val.shape, cv_val.Date.min().date(), "->", cv_val.Date.max().date())
print("Val holiday weeks:", cv_val.loc[cv_val.IsHoliday, "Date"].dt.strftime("%Y-%m-%d").unique().tolist())

CV train: (267184, 16) 2010-02-05 -> 2011-10-28
CV val  : (115856, 16) 2011-11-04 -> 2012-07-27
Val holiday weeks: ['2011-11-25', '2011-12-30', '2012-02-10']


In [5]:
MARKDOWN_COLS = [f"MarkDown{i}" for i in range(1, 6)]
FFILL_COLS = ["CPI", "Unemployment"]
MEDIAN_ONLY_COLS = ["Temperature", "Fuel_Price"]

class DataCleaner(BaseEstimator, TransformerMixin):

    def __init__(self, markdown_cols=None, ffill_cols=None, median_only_cols=None):
        self.markdown_cols = markdown_cols if markdown_cols is not None else MARKDOWN_COLS
        self.ffill_cols = ffill_cols if ffill_cols is not None else FFILL_COLS
        self.median_only_cols = median_only_cols if median_only_cols is not None else MEDIAN_ONLY_COLS

    def fit(self, X, y=None):
        X_sorted = X.sort_values(["Store", "Date"])
        filled = X_sorted.groupby("Store")[self.ffill_cols].ffill()
        self.last_known_ = filled.groupby(X_sorted["Store"]).last()
        self.medians_ = {c: X[c].median() for c in self.ffill_cols + self.median_only_cols}
        has_markdown = X[self.markdown_cols].notna().any(axis=1)
        self.markdown_cutoff_ = X.loc[has_markdown, "Date"].min() if has_markdown.any() else X["Date"].max()
        return self

    def transform(self, X):
        X = X.sort_values(["Store", "Date"]).copy()

        X[self.markdown_cols] = X[self.markdown_cols].fillna(0.0)
        X["markdown_recorded"] = (X["Date"] >= self.markdown_cutoff_).astype(int)

        for c in self.ffill_cols:
            X[c] = X.groupby("Store")[c].ffill()
            X[c] = X[c].fillna(X["Store"].map(self.last_known_[c]))
            X[c] = X[c].fillna(self.medians_[c])
        for c in self.median_only_cols:
            X[c] = X[c].fillna(self.medians_[c])

        return X.sort_index()

run = wandb.init(entity=WANDB_ENTITY, project=WANDB_PROJECT, group=WANDB_GROUP, name="LightGBM_Data_Cleaning", job_type="data-cleaning")

cleaner = DataCleaner()
cleaner.fit(cv_train)

cv_train_clean = cleaner.transform(cv_train)
cv_val_clean = cleaner.transform(cv_val)

missing_report = cv_train_clean[cleaner.markdown_cols + cleaner.ffill_cols + cleaner.median_only_cols].isna().mean()
print("Learned fill medians:", cleaner.medians_)
print("Learned markdown_recorded cutoff date:", cleaner.markdown_cutoff_.date())
print("\nRemaining missing % after cleaning:\n", missing_report)

wandb.config.update({"markdown_cutoff": str(cleaner.markdown_cutoff_.date()), **{f"median_{c}": v for c, v in cleaner.medians_.items()}})
wandb.log({"post_clean_missing_pct_total": float(missing_report.sum())})
run.finish()

wandb: setting up run en91qpyz


wandb: Tracking run with wandb version 0.28.0


wandb: Run data is saved locally in /Users/lizamarsagishvili/Downloads/final_project/wandb/run-20260709_215144-en91qpyz
wandb: Run `wandb offline` to turn off syncing.


wandb: Syncing run LightGBM_Data_Cleaning


wandb: ⭐️ View project at https://wandb.ai/toberi23-free-university-of-tbilisi-/Walmart-Recruiting-Store-Sales-Forecasting


...

wandb: uploading wandb-metadata.json; uploading requirements.txt; uploading wandb-summary.json; uploading config.yaml


wandb: uploading history steps 0-0, summary


wandb: 
wandb: Run history:
wandb: post_clean_missing_pct_total ▁
wandb: 
wandb: Run summary:
wandb: post_clean_missing_pct_total 0
wandb: 


wandb: 🚀 View run LightGBM_Data_Cleaning at: https://wandb.ai/toberi23-free-university-of-tbilisi-/Walmart-Recruiting-Store-Sales-Forecasting/runs/en91qpyz
wandb: ⭐️ View project at: https://wandb.ai/toberi23-free-university-of-tbilisi-/Walmart-Recruiting-Store-Sales-Forecasting
wandb: Synced 4 W&B file(s), 0 media file(s), 0 artifact file(s) and 0 other file(s)


wandb: Find logs at: ./wandb/run-20260709_215144-en91qpyz/logs


In [6]:
THANKSGIVING_DATES = pd.to_datetime(["2010-11-26", "2011-11-25", "2012-11-23", "2013-11-29"])
SUPERBOWL_DATES = pd.to_datetime(["2010-02-12", "2011-02-11", "2012-02-10", "2013-02-08"])
CHRISTMAS_DATES = pd.to_datetime(["2010-12-25", "2011-12-25", "2012-12-25", "2013-12-25"])
LAGS = [39, 52, 104]

def _signed_days_to_nearest(dates, anchors):
    arr = dates.values.astype("datetime64[D]")
    anc = np.asarray(anchors, dtype="datetime64[D]")
    diffs = (arr[:, None] - anc[None, :]).astype("timedelta64[D]").astype(int)
    idx = np.abs(diffs).argmin(axis=1)
    return diffs[np.arange(len(arr)), idx]

class TimeSeriesFeatureEngineer(BaseEstimator, TransformerMixin):

    def __init__(self, lags=None):
        self.lags = lags if lags is not None else LAGS

    def fit(self, X, y=None):
        self.history_ = X[["Store", "Dept", "Date", "Weekly_Sales"]].copy()
        self.store_dept_mean_ = X.groupby(["Store", "Dept"])["Weekly_Sales"].mean()
        self.dept_type_mean_ = X.groupby(["Dept", "Type"])["Weekly_Sales"].mean()
        self.global_mean_ = X["Weekly_Sales"].mean()
        woy = X["Date"].dt.isocalendar().week.astype(int)
        self.woy_mean_ = X.assign(weekofyear=woy).groupby(["Store", "Dept", "weekofyear"])["Weekly_Sales"].mean()
        self.hist_start_ = X.groupby(["Store", "Dept"])["Date"].min()
        self.store_categories_ = sorted(X["Store"].unique())
        self.dept_categories_ = sorted(X["Dept"].unique())
        self.type_categories_ = sorted(X["Type"].unique())
        return self

    def transform(self, X):
        X = X.copy()
        X["month"] = X["Date"].dt.month
        X["weekofyear"] = X["Date"].dt.isocalendar().week.astype(int)
        X["days_to_thanksgiving"] = _signed_days_to_nearest(X["Date"], THANKSGIVING_DATES)
        X["days_to_christmas"] = _signed_days_to_nearest(X["Date"], CHRISTMAS_DATES)
        X["days_to_superbowl"] = _signed_days_to_nearest(X["Date"], SUPERBOWL_DATES)

        for lag in self.lags:
            lag_tbl = self.history_.rename(columns={"Weekly_Sales": f"lag_{lag}"}).copy()
            lag_tbl["Date"] = lag_tbl["Date"] + pd.Timedelta(weeks=lag)
            X = X.merge(lag_tbl, on=["Store", "Dept", "Date"], how="left")

        sd_mean_tbl = self.store_dept_mean_.rename("store_dept_mean").reset_index()
        X = X.merge(sd_mean_tbl, on=["Store", "Dept"], how="left")
        X["store_dept_mean"] = X["store_dept_mean"].fillna(self.global_mean_)

        dt_mean_tbl = self.dept_type_mean_.rename("dept_type_mean").reset_index()
        X = X.merge(dt_mean_tbl, on=["Dept", "Type"], how="left")
        X["dept_type_mean"] = X["dept_type_mean"].fillna(self.global_mean_)

        woy_mean_tbl = self.woy_mean_.rename("sd_woy_mean").reset_index()
        X = X.merge(woy_mean_tbl, on=["Store", "Dept", "weekofyear"], how="left")
        X["sd_woy_mean"] = X["sd_woy_mean"].fillna(X["store_dept_mean"])

        hist_start_tbl = self.hist_start_.rename("_hist_start").reset_index()
        X = X.merge(hist_start_tbl, on=["Store", "Dept"], how="left")
        X["hist_len"] = ((X["Date"] - X["_hist_start"]).dt.days / 7.0).clip(lower=0)
        X["hist_len"] = X["hist_len"].fillna(0.0)
        X = X.drop(columns=["_hist_start"])

        X["Store"] = pd.Categorical(X["Store"], categories=self.store_categories_)
        X["Dept"] = pd.Categorical(X["Dept"], categories=self.dept_categories_)
        X["Type"] = pd.Categorical(X["Type"], categories=self.type_categories_)

        return X

run = wandb.init(entity=WANDB_ENTITY, project=WANDB_PROJECT, group=WANDB_GROUP, name="LightGBM_Feature_Engineering", job_type="feature-engineering", config={"lags": LAGS})

engineer = TimeSeriesFeatureEngineer().fit(cv_train_clean)
cv_train_feat = engineer.transform(cv_train_clean)
cv_val_feat = engineer.transform(cv_val_clean)

n_cold_start_val = cv_val_feat["lag_52"].isna().sum()
print("Engineered feature columns added:",
      [c for c in cv_train_feat.columns if c not in cv_train_clean.columns])
print(f"\nValidation rows with lag_52 undefined (short/cold-start series): {n_cold_start_val} / {len(cv_val_feat)}")
print(f"Mean hist_len (weeks) on validation rows: {cv_val_feat['hist_len'].mean():.1f}")

wandb.log({
    "n_engineered_features": len([c for c in cv_train_feat.columns if c not in cv_train_clean.columns]),
    "val_rows_lag52_undefined": int(n_cold_start_val),
    "val_mean_hist_len_weeks": float(cv_val_feat["hist_len"].mean()),
})
run.finish()

wandb: setting up run nzeeblq3


wandb: Tracking run with wandb version 0.28.0


wandb: Run data is saved locally in /Users/lizamarsagishvili/Downloads/final_project/wandb/run-20260709_215148-nzeeblq3
wandb: Run `wandb offline` to turn off syncing.


wandb: Syncing run LightGBM_Feature_Engineering


wandb: ⭐️ View project at https://wandb.ai/toberi23-free-university-of-tbilisi-/Walmart-Recruiting-Store-Sales-Forecasting


...

wandb: uploading wandb-metadata.json; uploading requirements.txt; uploading wandb-summary.json; uploading config.yaml


wandb: uploading history steps 0-0, summary


wandb: 
wandb: Run history:
wandb:    n_engineered_features ▁
wandb:  val_mean_hist_len_weeks ▁
wandb: val_rows_lag52_undefined ▁
wandb: 
wandb: Run summary:
wandb:    n_engineered_features 12
wandb:  val_mean_hist_len_weeks 108.8592
wandb: val_rows_lag52_undefined 3523
wandb: 


wandb: 🚀 View run LightGBM_Feature_Engineering at: https://wandb.ai/toberi23-free-university-of-tbilisi-/Walmart-Recruiting-Store-Sales-Forecasting/runs/nzeeblq3
wandb: ⭐️ View project at: https://wandb.ai/toberi23-free-university-of-tbilisi-/Walmart-Recruiting-Store-Sales-Forecasting
wandb: Synced 4 W&B file(s), 0 media file(s), 0 artifact file(s) and 0 other file(s)


wandb: Find logs at: ./wandb/run-20260709_215148-nzeeblq3/logs


In [7]:
CORE_FEATURES = [
    "Store", "Dept", "Type", "Size",
    "month", "weekofyear", "days_to_thanksgiving", "days_to_christmas", "days_to_superbowl",
    "lag_39", "lag_52", "lag_104",
    "store_dept_mean", "dept_type_mean", "sd_woy_mean", "hist_len",
]
EXOGENOUS_FEATURES = [
    "IsHoliday", "Temperature", "Fuel_Price", "CPI", "Unemployment",
    "MarkDown1", "MarkDown2", "MarkDown3", "MarkDown4", "MarkDown5", "markdown_recorded",
]
CATEGORICAL_FEATURES = ["Store", "Dept", "Type"]

class FeatureSelector(BaseEstimator, TransformerMixin):

    def __init__(self, core_features=None, exogenous_features=None):
        self.core_features = core_features if core_features is not None else CORE_FEATURES
        self.exogenous_features = exogenous_features if exogenous_features is not None else EXOGENOUS_FEATURES

    def fit(self, X, y, X_val=None, y_val=None):
        all_feats = self.core_features + self.exogenous_features
        model = lgb.LGBMRegressor(n_estimators=300, learning_rate=0.05, verbosity=-1, random_state=SEED)
        fit_kwargs = dict(categorical_feature=CATEGORICAL_FEATURES)
        if X_val is not None:
            fit_kwargs["eval_set"] = [(X_val[all_feats], y_val)]
            fit_kwargs["callbacks"] = [lgb.early_stopping(30, verbose=False)]
        model.fit(X[all_feats], y, **fit_kwargs)
        self.importances_ = pd.Series(model.feature_importances_, index=all_feats).sort_values(ascending=False)
        self.selected_features_ = list(self.importances_[self.importances_ > 0].index)
        return self

    def transform(self, X):
        return X[[c for c in self.selected_features_ if c in X.columns]]

run = wandb.init(entity=WANDB_ENTITY, project=WANDB_PROJECT, group=WANDB_GROUP, name="LightGBM_Feature_Selection", job_type="feature-selection")

selector = FeatureSelector()
selector.fit(
    cv_train_feat, cv_train_feat["Weekly_Sales"],
    X_val=cv_val_feat, y_val=cv_val_feat["Weekly_Sales"],
)

print("Feature importances (all candidates), ranked:\n", selector.importances_)
n_dropped = len(selector.core_features + selector.exogenous_features) - len(selector.selected_features_)
print(f"\nFeatures with zero importance (dropped): {n_dropped}")
print("Selected features:", selector.selected_features_)

wandb.log({
    "n_candidate_features": len(selector.core_features + selector.exogenous_features),
    "n_selected_features": len(selector.selected_features_),
    **{f"importance_{c}": float(v) for c, v in selector.importances_.items()},
})
run.finish()

wandb: setting up run g4erfosh


wandb: Tracking run with wandb version 0.28.0


wandb: Run data is saved locally in /Users/lizamarsagishvili/Downloads/final_project/wandb/run-20260709_215152-g4erfosh
wandb: Run `wandb offline` to turn off syncing.


wandb: Syncing run LightGBM_Feature_Selection


wandb: ⭐️ View project at https://wandb.ai/toberi23-free-university-of-tbilisi-/Walmart-Recruiting-Store-Sales-Forecasting


...

wandb: uploading wandb-metadata.json; uploading requirements.txt; uploading wandb-summary.json; uploading config.yaml


wandb: uploading history steps 0-0, summary


wandb: 
wandb: Run history:
wandb:        importance_CPI ▁
wandb:       importance_Dept ▁
wandb: importance_Fuel_Price ▁
wandb:  importance_IsHoliday ▁
wandb:  importance_MarkDown1 ▁
wandb:  importance_MarkDown2 ▁
wandb:  importance_MarkDown3 ▁
wandb:  importance_MarkDown4 ▁
wandb:  importance_MarkDown5 ▁
wandb:       importance_Size ▁
wandb:                   +19 ...
wandb: 
wandb: Run summary:
wandb:        importance_CPI 294
wandb:       importance_Dept 887
wandb: importance_Fuel_Price 364
wandb:  importance_IsHoliday 88
wandb:  importance_MarkDown1 0
wandb:  importance_MarkDown2 0
wandb:  importance_MarkDown3 0
wandb:  importance_MarkDown4 0
wandb:  importance_MarkDown5 0
wandb:       importance_Size 102
wandb:                   +19 ...
wandb: 


wandb: 🚀 View run LightGBM_Feature_Selection at: https://wandb.ai/toberi23-free-university-of-tbilisi-/Walmart-Recruiting-Store-Sales-Forecasting/runs/g4erfosh
wandb: ⭐️ View project at: https://wandb.ai/toberi23-free-university-of-tbilisi-/Walmart-Recruiting-Store-Sales-Forecasting
wandb: Synced 4 W&B file(s), 0 media file(s), 0 artifact file(s) and 0 other file(s)


wandb: Find logs at: ./wandb/run-20260709_215152-g4erfosh/logs


In [8]:
CHRISTMAS_WEEK52_ANCHORS = {
    2010: pd.Timestamp("2010-12-31"),
    2011: pd.Timestamp("2011-12-30"),
    2012: pd.Timestamp("2012-12-28"),
}
TARGET_YEAR = 2012

class ChristmasWeekShiftAdjuster:

    def __init__(self, bulge_threshold=1.10, christmas_anchors=None, target_year=TARGET_YEAR):
        self.bulge_threshold = bulge_threshold
        self.christmas_anchors = christmas_anchors if christmas_anchors is not None else CHRISTMAS_WEEK52_ANCHORS
        self.target_year = target_year

    def fit(self, train_df):
        self.train_start_ = train_df.groupby(["Store", "Dept"])["Date"].min()
        self.dec_cutoffs_ = {year: anchor - pd.Timedelta(weeks=4) for year, anchor in self.christmas_anchors.items()}
        return self

    def adjust(self, X, preds):
        pred_df = X[["Store", "Dept", "Date"]].copy()
        pred_df["Weekly_Sales"] = preds
        pred_df["_row"] = np.arange(len(pred_df))

        adjusted = preds.copy()
        self.last_stats_ = {"n_bulge_departments": 0, "n_shifted_two_year": 0,
                             "n_shifted_one_year": 0, "total_abs_sales_shifted": 0.0}

        for year, anchor in self.christmas_anchors.items():
            if year != self.target_year:
                continue
            week_dates = {w: anchor - pd.Timedelta(weeks=52 - w) for w in [48, 49, 50, 51, 52]}
            week_lookup = {v: k for k, v in week_dates.items()}
            in_window = pred_df["Date"].isin(week_lookup)
            if not in_window.any():
                continue

            sub = pred_df.loc[in_window].copy()
            sub["week"] = sub["Date"].map(week_lookup)
            wide = sub.pivot_table(index=["Store", "Dept"], columns="week", values="Weekly_Sales")
            wide = wide.dropna(subset=[48, 49, 50, 51, 52])
            if wide.empty:
                continue

            avg_49_51 = wide[[49, 50, 51]].mean(axis=1)
            avg_48_52 = wide[[48, 52]].mean(axis=1)
            has_bulge = avg_49_51 >= self.bulge_threshold * avg_48_52

            train_start = self.train_start_.reindex(wide.index)
            has_two_years = train_start <= self.dec_cutoffs_.get(year - 1, pd.Timestamp.min)
            has_one_year = (train_start <= self.dec_cutoffs_.get(year, pd.Timestamp.min)) & (~has_two_years)

            shift_fraction = pd.Series(0.0, index=wide.index)
            shift_fraction[has_two_years] = 2.5 / 7
            shift_fraction[has_one_year] = 2.0 / 7
            shift_fraction[~has_bulge] = 0.0

            to_shift = shift_fraction[shift_fraction > 0].index
            if len(to_shift) == 0:
                continue

            shifted = wide.copy()
            for key in to_shift:
                vals = wide.loc[key, [48, 49, 50, 51, 52]].to_numpy(dtype=float)
                s = shift_fraction[key]
                shifted.loc[key] = [(1 - s) * vals[i] + s * vals[(i - 1) % 5] for i in range(5)]

            shifted_long = shifted.loc[to_shift].stack().rename("Weekly_Sales_shifted").reset_index()
            shifted_long["Date"] = shifted_long["week"].map(week_dates)
            merged = sub.merge(shifted_long[["Store", "Dept", "Date", "Weekly_Sales_shifted"]],
                                on=["Store", "Dept", "Date"], how="left")
            merged = merged.dropna(subset=["Weekly_Sales_shifted"])
            adjusted[merged["_row"].to_numpy()] = merged["Weekly_Sales_shifted"].to_numpy()

            delta_abs = (shifted.loc[to_shift] - wide.loc[to_shift]).abs().to_numpy().sum()
            self.last_stats_["n_bulge_departments"] += int(has_bulge.sum())
            self.last_stats_["n_shifted_two_year"] += int((shift_fraction == 2.5 / 7).sum())
            self.last_stats_["n_shifted_one_year"] += int((shift_fraction == 2.0 / 7).sum())
            self.last_stats_["total_abs_sales_shifted"] += float(delta_abs)

        return adjusted

DEFAULT_LGBM_PARAMS = dict(
    num_leaves=31, max_depth=-1, learning_rate=0.1, n_estimators=100,
    min_child_samples=20, colsample_bytree=1.0, subsample=1.0, subsample_freq=0,
    reg_alpha=0.0, reg_lambda=0.0, objective="regression",
    use_exogenous=True, use_sample_weight=False, use_christmas_shift=False,
)

class LGBMForecastPipeline(BaseEstimator, RegressorMixin):

    def __init__(self, num_leaves=31, max_depth=-1, learning_rate=0.1, n_estimators=100,
                 min_child_samples=20, colsample_bytree=1.0, subsample=1.0, subsample_freq=0,
                 reg_alpha=0.0, reg_lambda=0.0, objective="regression",
                 use_exogenous=True, use_sample_weight=False,
                 use_christmas_shift=False, christmas_bulge_threshold=1.10,
                 early_stopping_weeks=8, early_stopping_rounds=50):
        self.num_leaves = num_leaves
        self.max_depth = max_depth
        self.learning_rate = learning_rate
        self.n_estimators = n_estimators
        self.min_child_samples = min_child_samples
        self.colsample_bytree = colsample_bytree
        self.subsample = subsample
        self.subsample_freq = subsample_freq
        self.reg_alpha = reg_alpha
        self.reg_lambda = reg_lambda
        self.objective = objective
        self.use_exogenous = use_exogenous
        self.use_sample_weight = use_sample_weight
        self.use_christmas_shift = use_christmas_shift
        self.christmas_bulge_threshold = christmas_bulge_threshold
        self.early_stopping_weeks = early_stopping_weeks
        self.early_stopping_rounds = early_stopping_rounds

    def _feature_list(self):
        return CORE_FEATURES + (EXOGENOUS_FEATURES if self.use_exogenous else [])

    def fit(self, X, y=None):
        self.cleaner_ = DataCleaner().fit(X)
        X_clean = self.cleaner_.transform(X)

        cutoff = X_clean["Date"].max() - pd.Timedelta(weeks=self.early_stopping_weeks)
        fit_part = X_clean[X_clean["Date"] <= cutoff]
        es_part = X_clean[X_clean["Date"] > cutoff]

        fit_engineer = TimeSeriesFeatureEngineer().fit(fit_part)
        fit_feat = fit_engineer.transform(fit_part)
        es_feat = fit_engineer.transform(es_part) if len(es_part) else None

        feats = self._feature_list()
        cat_cols = [c for c in CATEGORICAL_FEATURES if c in feats]
        weight = np.where(fit_feat["IsHoliday"], 5.0, 1.0) if self.use_sample_weight else None

        self.model_ = lgb.LGBMRegressor(
            num_leaves=self.num_leaves, max_depth=self.max_depth, learning_rate=self.learning_rate,
            n_estimators=self.n_estimators, min_child_samples=self.min_child_samples,
            colsample_bytree=self.colsample_bytree, subsample=self.subsample,
            subsample_freq=self.subsample_freq, reg_alpha=self.reg_alpha, reg_lambda=self.reg_lambda,
            objective=self.objective, verbosity=-1, random_state=SEED,
        )
        fit_kwargs = dict(categorical_feature=cat_cols)
        if es_feat is not None and len(es_feat):
            fit_kwargs["eval_set"] = [(es_feat[feats], es_feat["Weekly_Sales"])]
            fit_kwargs["callbacks"] = [lgb.early_stopping(self.early_stopping_rounds, verbose=False)]
        self.model_.fit(fit_feat[feats], fit_feat["Weekly_Sales"], sample_weight=weight, **fit_kwargs)

        self.engineer_ = TimeSeriesFeatureEngineer().fit(X_clean)

        if self.use_christmas_shift:
            self.christmas_adjuster_ = ChristmasWeekShiftAdjuster(
                bulge_threshold=self.christmas_bulge_threshold
            ).fit(X_clean)
        return self

    def predict(self, X):
        X_clean = self.cleaner_.transform(X)
        X_feat = self.engineer_.transform(X_clean)
        feats = self._feature_list()
        preds = self.model_.predict(X_feat[feats])

        if self.use_christmas_shift:
            preds = self.christmas_adjuster_.adjust(X_feat, preds)
        return preds

def evaluate_config(config, train_full, val_full, verbose=False):
    pipeline = LGBMForecastPipeline(**config)
    pipeline.fit(train_full)
    preds = pipeline.predict(val_full)
    score = wmae(val_full["Weekly_Sales"], preds, val_full["IsHoliday"])
    n_trees = getattr(pipeline.model_, "best_iteration_", None) or pipeline.model_.n_estimators_
    if verbose:
        print(f"WMAE: {score:.2f}  (trees used: {n_trees})")
    return score, n_trees

def run_stage(stage_name, configs, base_config):
    results = []
    for cfg_overrides in configs:
        cfg = {**base_config, **cfg_overrides}
        tag = "_".join(f"{k}{v}" for k, v in cfg_overrides.items())
        run_name = f"LightGBM_Training_Sweep_{stage_name}_{tag}"

        run = wandb.init(entity=WANDB_ENTITY, project=WANDB_PROJECT, group=WANDB_GROUP, name=run_name, job_type="training-sweep",
                          config={**cfg, "stage": stage_name}, reinit=True)
        t0 = time.time()
        score, n_trees = evaluate_config(cfg, cv_train, cv_val)
        elapsed = time.time() - t0
        wandb.log({"wmae": score, "seconds": elapsed, "n_trees": int(n_trees)})
        run.finish()

        results.append({**cfg_overrides, "wmae": score, "seconds": elapsed, "n_trees": n_trees})
        print(f"{run_name}: WMAE={score:.2f} trees={n_trees} ({elapsed:.1f}s)")

    return pd.DataFrame(results).sort_values("wmae").reset_index(drop=True)

In [9]:
run = wandb.init(entity=WANDB_ENTITY, project=WANDB_PROJECT, group=WANDB_GROUP, name="LightGBM_Training_Baseline", job_type="training",
                  config={**DEFAULT_LGBM_PARAMS, "fold": "B_calendar_aligned"})

t0 = time.time()
baseline_wmae, baseline_trees = evaluate_config(DEFAULT_LGBM_PARAMS, cv_train, cv_val, verbose=True)
elapsed = time.time() - t0

wandb.log({"wmae": baseline_wmae, "seconds": elapsed, "n_trees": int(baseline_trees)})
print(f"Baseline WMAE: {baseline_wmae:.2f} in {elapsed:.1f}s")
run.finish()

wandb: setting up run 2n8sve96


wandb: Tracking run with wandb version 0.28.0


wandb: Run data is saved locally in /Users/lizamarsagishvili/Downloads/final_project/wandb/run-20260709_215158-2n8sve96
wandb: Run `wandb offline` to turn off syncing.


wandb: Syncing run LightGBM_Training_Baseline


wandb: ⭐️ View project at https://wandb.ai/toberi23-free-university-of-tbilisi-/Walmart-Recruiting-Store-Sales-Forecasting


...

wandb: uploading requirements.txt; uploading wandb-summary.json


wandb: uploading history steps 0-0, summary


wandb: 
wandb: Run history:
wandb: n_trees ▁
wandb: seconds ▁
wandb:    wmae ▁
wandb: 
wandb: Run summary:
wandb: n_trees 95
wandb: seconds 1.28901
wandb:    wmae 2250.52598
wandb: 


wandb: 🚀 View run LightGBM_Training_Baseline at: https://wandb.ai/toberi23-free-university-of-tbilisi-/Walmart-Recruiting-Store-Sales-Forecasting/runs/2n8sve96
wandb: ⭐️ View project at: https://wandb.ai/toberi23-free-university-of-tbilisi-/Walmart-Recruiting-Store-Sales-Forecasting
wandb: Synced 4 W&B file(s), 0 media file(s), 0 artifact file(s) and 0 other file(s)


wandb: Find logs at: ./wandb/run-20260709_215158-2n8sve96/logs


In [10]:
stage1_configs = [{"num_leaves": v} for v in [15, 31, 63, 127, 255]]
stage1_results = run_stage("num_leaves", stage1_configs, DEFAULT_LGBM_PARAMS)
stage1_results

wandb: WARNING Using a boolean value for 'reinit' is deprecated. Use 'return_previous' or 'finish_previous' instead.


wandb: setting up run dqsb2taw


wandb: Tracking run with wandb version 0.28.0


wandb: Run data is saved locally in /Users/lizamarsagishvili/Downloads/final_project/wandb/run-20260709_215204-dqsb2taw
wandb: Run `wandb offline` to turn off syncing.


wandb: Syncing run LightGBM_Training_Sweep_num_leaves_num_leaves15


...

wandb: 
wandb: Run history:
wandb: n_trees ▁
wandb: seconds ▁
wandb:    wmae ▁
wandb: 
wandb: Run summary:
wandb: n_trees 85
wandb: seconds 4.0016
wandb:    wmae 2382.7597
wandb: 


wandb: 🚀 View run LightGBM_Training_Sweep_num_leaves_num_leaves255 at: https://wandb.ai/toberi23-free-university-of-tbilisi-/Walmart-Recruiting-Store-Sales-Forecasting/runs/dvbwnkas
wandb: ⭐️ View project at: https://wandb.ai/toberi23-free-university-of-tbilisi-/Walmart-Recruiting-Store-Sales-Forecasting
wandb: Synced 4 W&B file(s), 0 media file(s), 0 artifact file(s) and 0 other file(s)


wandb: Find logs at: ./wandb/run-20260709_215225-dvbwnkas/logs


LightGBM_Training_Sweep_num_leaves_num_leaves255: WMAE=2382.76 trees=85 (4.0s)


,num_leaves,wmae,seconds,n_trees
0,15,2233.789615,0.985065,99
1,31,2250.525979,1.237792,95
2,63,2299.836108,1.674197,100
3,127,2350.307876,2.474392,87
4,255,2382.759699,4.001599,85


In [11]:
best_num_leaves = int(stage1_results.loc[0, "num_leaves"])
config_after_stage1 = {**DEFAULT_LGBM_PARAMS, "num_leaves": best_num_leaves}

stage2_configs = [{"max_depth": v} for v in [-1, 4, 6, 8, 10]]
stage2_results = run_stage("max_depth", stage2_configs, config_after_stage1)
stage2_results

wandb: setting up run d55n3z5f


wandb: Tracking run with wandb version 0.28.0


wandb: Run data is saved locally in /Users/lizamarsagishvili/Downloads/final_project/wandb/run-20260709_215232-d55n3z5f
wandb: Run `wandb offline` to turn off syncing.


wandb: Syncing run LightGBM_Training_Sweep_max_depth_max_depth-1


wandb: ⭐️ View project at https://wandb.ai/toberi23-free-university-of-tbilisi-/Walmart-Recruiting-Store-Sales-Forecasting


...

wandb: 
wandb: Run history:
wandb: n_trees ▁
wandb: seconds ▁
wandb:    wmae ▁
wandb: 
wandb: Run summary:
wandb: n_trees 100
wandb: seconds 1.02059
wandb:    wmae 2234.58266
wandb: 


wandb: 🚀 View run LightGBM_Training_Sweep_max_depth_max_depth10 at: https://wandb.ai/toberi23-free-university-of-tbilisi-/Walmart-Recruiting-Store-Sales-Forecasting/runs/phrt1nt3
wandb: ⭐️ View project at: https://wandb.ai/toberi23-free-university-of-tbilisi-/Walmart-Recruiting-Store-Sales-Forecasting
wandb: Synced 4 W&B file(s), 0 media file(s), 0 artifact file(s) and 0 other file(s)


wandb: Find logs at: ./wandb/run-20260709_215251-phrt1nt3/logs


LightGBM_Training_Sweep_max_depth_max_depth10: WMAE=2234.58 trees=100 (1.0s)


,max_depth,wmae,seconds,n_trees
0,-1,2233.789615,1.026334,99
1,10,2234.582659,1.020589,100
2,6,2237.318956,1.011338,100
3,8,2241.979005,1.014091,100
4,4,2248.624542,0.975666,100


In [12]:
best_max_depth = int(stage2_results.loc[0, "max_depth"])
config_after_stage2 = {**config_after_stage1, "max_depth": best_max_depth}

stage3_configs = [{"learning_rate": v, "n_estimators": 3000} for v in [0.005, 0.01, 0.03, 0.05, 0.1]]
stage3_results = run_stage("learning_rate", stage3_configs, config_after_stage2)
stage3_results

wandb: setting up run wfjj4vsw


wandb: Tracking run with wandb version 0.28.0


wandb: Run data is saved locally in /Users/lizamarsagishvili/Downloads/final_project/wandb/run-20260709_215256-wfjj4vsw
wandb: Run `wandb offline` to turn off syncing.


wandb: Syncing run LightGBM_Training_Sweep_learning_rate_learning_rate0.005_n_estimators3000


wandb: ⭐️ View project at https://wandb.ai/toberi23-free-university-of-tbilisi-/Walmart-Recruiting-Store-Sales-Forecasting


...

wandb: 
wandb: Run history:
wandb: n_trees ▁
wandb: seconds ▁
wandb:    wmae ▁
wandb: 
wandb: Run summary:
wandb: n_trees 345
wandb: seconds 1.91701
wandb:    wmae 2229.94428
wandb: 


wandb: 🚀 View run LightGBM_Training_Sweep_learning_rate_learning_rate0.1_n_estimators3000 at: https://wandb.ai/toberi23-free-university-of-tbilisi-/Walmart-Recruiting-Store-Sales-Forecasting/runs/62d95fh9
wandb: ⭐️ View project at: https://wandb.ai/toberi23-free-university-of-tbilisi-/Walmart-Recruiting-Store-Sales-Forecasting
wandb: Synced 4 W&B file(s), 0 media file(s), 0 artifact file(s) and 0 other file(s)


wandb: Find logs at: ./wandb/run-20260709_215344-62d95fh9/logs


LightGBM_Training_Sweep_learning_rate_learning_rate0.1_n_estimators3000: WMAE=2229.94 trees=345 (1.9s)


,learning_rate,n_estimators,wmae,seconds,n_trees
0,0.050,3000,2199.385506,5.966181,770
1,0.030,3000,2219.156917,7.512523,1062
2,0.010,3000,2222.168758,6.719103,1650
3,0.005,3000,2224.393640,11.846538,3000
4,0.100,3000,2229.944280,1.917010,345


In [13]:
best_lr = stage3_results.loc[0, "learning_rate"]
config_after_stage3 = {**config_after_stage2, "learning_rate": best_lr, "n_estimators": 3000}

stage4_configs = [{"min_child_samples": v} for v in [5, 10, 20, 50, 100]]
stage4_results = run_stage("min_child_samples", stage4_configs, config_after_stage3)
stage4_results

wandb: setting up run 4nkgff7o


wandb: Tracking run with wandb version 0.28.0


wandb: Run data is saved locally in /Users/lizamarsagishvili/Downloads/final_project/wandb/run-20260709_215351-4nkgff7o
wandb: Run `wandb offline` to turn off syncing.


wandb: Syncing run LightGBM_Training_Sweep_min_child_samples_min_child_samples5


wandb: ⭐️ View project at https://wandb.ai/toberi23-free-university-of-tbilisi-/Walmart-Recruiting-Store-Sales-Forecasting


...

wandb: 
wandb: Run history:
wandb: n_trees ▁
wandb: seconds ▁
wandb:    wmae ▁
wandb: 
wandb: Run summary:
wandb: n_trees 481
wandb: seconds 2.51003
wandb:    wmae 2212.25522
wandb: 


wandb: 🚀 View run LightGBM_Training_Sweep_min_child_samples_min_child_samples100 at: https://wandb.ai/toberi23-free-university-of-tbilisi-/Walmart-Recruiting-Store-Sales-Forecasting/runs/d4j16qyb
wandb: ⭐️ View project at: https://wandb.ai/toberi23-free-university-of-tbilisi-/Walmart-Recruiting-Store-Sales-Forecasting
wandb: Synced 4 W&B file(s), 0 media file(s), 0 artifact file(s) and 0 other file(s)


wandb: Find logs at: ./wandb/run-20260709_215420-d4j16qyb/logs


LightGBM_Training_Sweep_min_child_samples_min_child_samples100: WMAE=2212.26 trees=481 (2.5s)


,min_child_samples,wmae,seconds,n_trees
0,20,2199.385506,3.339111,770
1,100,2212.255217,2.510031,481
2,50,2219.278629,1.758181,277
3,10,2232.790531,3.089269,702
4,5,2233.834977,3.297135,743


In [14]:
best_mcs = int(stage4_results.loc[0, "min_child_samples"])
config_after_stage4 = {**config_after_stage3, "min_child_samples": best_mcs}

stage5_configs = [
    {"colsample_bytree": cb, "subsample": sb, "subsample_freq": 1 if sb < 1.0 else 0}
    for cb in [0.6, 0.8, 1.0]
    for sb in [0.6, 0.8, 1.0]
]
stage5_results = run_stage("colsample_subsample", stage5_configs, config_after_stage4)
stage5_results.head(10)

wandb: setting up run mw15kt60


wandb: Tracking run with wandb version 0.28.0


wandb: Run data is saved locally in /Users/lizamarsagishvili/Downloads/final_project/wandb/run-20260709_215427-mw15kt60
wandb: Run `wandb offline` to turn off syncing.


wandb: Syncing run LightGBM_Training_Sweep_colsample_subsample_colsample_bytree0.6_subsample0.6_subsample_freq1


wandb: ⭐️ View project at https://wandb.ai/toberi23-free-university-of-tbilisi-/Walmart-Recruiting-Store-Sales-Forecasting


...

wandb: 
wandb: Run history:
wandb: n_trees ▁
wandb: seconds ▁
wandb:    wmae ▁
wandb: 
wandb: Run summary:
wandb: n_trees 770
wandb: seconds 3.44753
wandb:    wmae 2199.38551
wandb: 


wandb: 🚀 View run LightGBM_Training_Sweep_colsample_subsample_colsample_bytree1.0_subsample1.0_subsample_freq0 at: https://wandb.ai/toberi23-free-university-of-tbilisi-/Walmart-Recruiting-Store-Sales-Forecasting/runs/hl6p8kyi
wandb: ⭐️ View project at: https://wandb.ai/toberi23-free-university-of-tbilisi-/Walmart-Recruiting-Store-Sales-Forecasting
wandb: Synced 4 W&B file(s), 0 media file(s), 0 artifact file(s) and 0 other file(s)


wandb: Find logs at: ./wandb/run-20260709_215515-hl6p8kyi/logs


LightGBM_Training_Sweep_colsample_subsample_colsample_bytree1.0_subsample1.0_subsample_freq0: WMAE=2199.39 trees=770 (3.4s)


,colsample_bytree,subsample,subsample_freq,wmae,seconds,n_trees
0,0.6,1.0,0,2113.376577,2.374730,343
1,0.6,0.8,1,2124.752160,2.154256,328
2,0.6,0.6,1,2141.198995,1.832189,245
3,0.8,0.8,1,2163.532489,2.856968,593
4,0.8,0.6,1,2176.880326,1.830442,295
5,1.0,0.8,1,2187.478886,2.738008,538
6,1.0,0.6,1,2188.284092,2.792411,388
7,0.8,1.0,0,2188.518388,1.999801,339
8,1.0,1.0,0,2199.385506,3.447529,770


In [15]:
best_cb = stage5_results.loc[0, "colsample_bytree"]
best_sb = stage5_results.loc[0, "subsample"]
best_sbf = int(stage5_results.loc[0, "subsample_freq"])
config_after_stage5 = {**config_after_stage4, "colsample_bytree": best_cb, "subsample": best_sb, "subsample_freq": best_sbf}

stage6_configs = [
    {"reg_alpha": ra, "reg_lambda": rl}
    for ra in [0.0, 0.1, 1.0]
    for rl in [0.0, 0.1, 1.0]
]
stage6_results = run_stage("reg_alpha_lambda", stage6_configs, config_after_stage5)
stage6_results.head(10)

wandb: setting up run d6r3oo9f


wandb: Tracking run with wandb version 0.28.0


wandb: Run data is saved locally in /Users/lizamarsagishvili/Downloads/final_project/wandb/run-20260709_215522-d6r3oo9f
wandb: Run `wandb offline` to turn off syncing.


wandb: Syncing run LightGBM_Training_Sweep_reg_alpha_lambda_reg_alpha0.0_reg_lambda0.0


wandb: ⭐️ View project at https://wandb.ai/toberi23-free-university-of-tbilisi-/Walmart-Recruiting-Store-Sales-Forecasting


...

wandb: 
wandb: Run history:
wandb: n_trees ▁
wandb: seconds ▁
wandb:    wmae ▁
wandb: 
wandb: Run summary:
wandb: n_trees 359
wandb: seconds 2.33434
wandb:    wmae 2127.15129
wandb: 


wandb: 🚀 View run LightGBM_Training_Sweep_reg_alpha_lambda_reg_alpha1.0_reg_lambda1.0 at: https://wandb.ai/toberi23-free-university-of-tbilisi-/Walmart-Recruiting-Store-Sales-Forecasting/runs/79bfhl4o
wandb: ⭐️ View project at: https://wandb.ai/toberi23-free-university-of-tbilisi-/Walmart-Recruiting-Store-Sales-Forecasting
wandb: Synced 4 W&B file(s), 0 media file(s), 0 artifact file(s) and 0 other file(s)


wandb: Find logs at: ./wandb/run-20260709_215617-79bfhl4o/logs


LightGBM_Training_Sweep_reg_alpha_lambda_reg_alpha1.0_reg_lambda1.0: WMAE=2127.15 trees=359 (2.3s)


,reg_alpha,reg_lambda,wmae,seconds,n_trees
0,1.0,0.0,2113.376560,2.386246,343
1,0.1,0.0,2113.376575,2.243405,343
2,0.0,0.0,2113.376577,2.289456,343
3,1.0,0.1,2123.141465,3.788870,350
4,0.1,0.1,2123.141480,5.149621,350
5,0.0,0.1,2123.141482,2.248685,350
6,1.0,1.0,2127.151292,2.334343,359
7,0.1,1.0,2127.151336,2.348929,359
8,0.0,1.0,2127.151341,3.920342,359


In [16]:
best_ra = stage6_results.loc[0, "reg_alpha"]
best_rl = stage6_results.loc[0, "reg_lambda"]
config_after_stage6 = {**config_after_stage5, "reg_alpha": best_ra, "reg_lambda": best_rl}

stage7_configs = [{"objective": v} for v in ["regression", "regression_l1", "huber"]]
stage7_results = run_stage("objective", stage7_configs, config_after_stage6)
stage7_results

wandb: setting up run bqw4lq54


wandb: Tracking run with wandb version 0.28.0


wandb: Run data is saved locally in /Users/lizamarsagishvili/Downloads/final_project/wandb/run-20260709_215623-bqw4lq54
wandb: Run `wandb offline` to turn off syncing.


wandb: Syncing run LightGBM_Training_Sweep_objective_objectiveregression


wandb: ⭐️ View project at https://wandb.ai/toberi23-free-university-of-tbilisi-/Walmart-Recruiting-Store-Sales-Forecasting


...

wandb: 
wandb: Run history:
wandb: n_trees ▁
wandb: seconds ▁
wandb:    wmae ▁
wandb: 
wandb: Run summary:
wandb: n_trees 3000
wandb: seconds 13.63115
wandb:    wmae 15677.29438
wandb: 


wandb: 🚀 View run LightGBM_Training_Sweep_objective_objectivehuber at: https://wandb.ai/toberi23-free-university-of-tbilisi-/Walmart-Recruiting-Store-Sales-Forecasting/runs/mgwmayn9
wandb: ⭐️ View project at: https://wandb.ai/toberi23-free-university-of-tbilisi-/Walmart-Recruiting-Store-Sales-Forecasting
wandb: Synced 4 W&B file(s), 0 media file(s), 0 artifact file(s) and 0 other file(s)


wandb: Find logs at: ./wandb/run-20260709_215641-mgwmayn9/logs


LightGBM_Training_Sweep_objective_objectivehuber: WMAE=15677.29 trees=3000 (13.6s)


,objective,wmae,seconds,n_trees
0,regression,2113.376560,5.057652,343
1,regression_l1,2247.624076,5.047718,541
2,huber,15677.294383,13.631149,3000


In [17]:
best_objective = stage7_results.loc[0, "objective"]
config_after_stage7 = {**config_after_stage6, "objective": best_objective}

stage8_configs = [{"use_sample_weight": v} for v in [True, False]]
stage8_results = run_stage("use_sample_weight", stage8_configs, config_after_stage7)
stage8_results

wandb: setting up run nyugp3f4


wandb: Tracking run with wandb version 0.28.0


wandb: Run data is saved locally in /Users/lizamarsagishvili/Downloads/final_project/wandb/run-20260709_215658-nyugp3f4
wandb: Run `wandb offline` to turn off syncing.


wandb: Syncing run LightGBM_Training_Sweep_use_sample_weight_use_sample_weightTrue


wandb: ⭐️ View project at https://wandb.ai/toberi23-free-university-of-tbilisi-/Walmart-Recruiting-Store-Sales-Forecasting


...

wandb: 
wandb: Run history:
wandb: n_trees ▁
wandb: seconds ▁
wandb:    wmae ▁
wandb: 
wandb: Run summary:
wandb: n_trees 343
wandb: seconds 2.34842
wandb:    wmae 2113.37656
wandb: 


wandb: 🚀 View run LightGBM_Training_Sweep_use_sample_weight_use_sample_weightFalse at: https://wandb.ai/toberi23-free-university-of-tbilisi-/Walmart-Recruiting-Store-Sales-Forecasting/runs/zj2sfy1p
wandb: ⭐️ View project at: https://wandb.ai/toberi23-free-university-of-tbilisi-/Walmart-Recruiting-Store-Sales-Forecasting
wandb: Synced 4 W&B file(s), 0 media file(s), 0 artifact file(s) and 0 other file(s)


wandb: Find logs at: ./wandb/run-20260709_215706-zj2sfy1p/logs


LightGBM_Training_Sweep_use_sample_weight_use_sample_weightFalse: WMAE=2113.38 trees=343 (2.3s)


,use_sample_weight,wmae,seconds,n_trees
0,False,2113.376560,2.348424,343
1,True,2131.780837,4.296787,260


In [18]:
best_usw = bool(stage8_results.loc[0, "use_sample_weight"])
config_after_stage8 = {**config_after_stage7, "use_sample_weight": best_usw}

stage9_configs = [{"use_exogenous": v} for v in [True, False]]
stage9_results = run_stage("use_exogenous", stage9_configs, config_after_stage8)
stage9_results

wandb: setting up run 4uf2yo3a


wandb: Tracking run with wandb version 0.28.0


wandb: Run data is saved locally in /Users/lizamarsagishvili/Downloads/final_project/wandb/run-20260709_215713-4uf2yo3a
wandb: Run `wandb offline` to turn off syncing.


wandb: Syncing run LightGBM_Training_Sweep_use_exogenous_use_exogenousTrue


wandb: ⭐️ View project at https://wandb.ai/toberi23-free-university-of-tbilisi-/Walmart-Recruiting-Store-Sales-Forecasting


...

wandb: 
wandb: Run history:
wandb: n_trees ▁
wandb: seconds ▁
wandb:    wmae ▁
wandb: 
wandb: Run summary:
wandb: n_trees 355
wandb: seconds 2.1774
wandb:    wmae 2125.19023
wandb: 


wandb: 🚀 View run LightGBM_Training_Sweep_use_exogenous_use_exogenousFalse at: https://wandb.ai/toberi23-free-university-of-tbilisi-/Walmart-Recruiting-Store-Sales-Forecasting/runs/x491cgka
wandb: ⭐️ View project at: https://wandb.ai/toberi23-free-university-of-tbilisi-/Walmart-Recruiting-Store-Sales-Forecasting
wandb: Synced 4 W&B file(s), 0 media file(s), 0 artifact file(s) and 0 other file(s)


wandb: Find logs at: ./wandb/run-20260709_215721-x491cgka/logs


LightGBM_Training_Sweep_use_exogenous_use_exogenousFalse: WMAE=2125.19 trees=355 (2.2s)


,use_exogenous,wmae,seconds,n_trees
0,True,2113.376560,4.024433,343
1,False,2125.190229,2.177401,355


In [19]:
def describe_stage(name, results_df, param_cols, hypothesis):
    best_idx = 0
    worst_idx = len(results_df) - 1
    best_vals = {c: results_df.loc[best_idx, c] for c in param_cols}
    worst_vals = {c: results_df.loc[worst_idx, c] for c in param_cols}
    best_wmae = results_df.loc[best_idx, "wmae"]
    worst_wmae = results_df.loc[worst_idx, "wmae"]
    spread = worst_wmae - best_wmae
    print(f"--- {name} ---")
    print(f"Best:  {best_vals} -> WMAE={best_wmae:.2f}")
    print(f"Worst: {worst_vals} -> WMAE={worst_wmae:.2f}")
    print(f"Spread across this stage: {spread:.2f} WMAE ({'material' if spread > 0.02*best_wmae else 'small'} relative to best score)")
    print(f"Hypothesis was: {hypothesis}")
    print(f"Consistent with hypothesis direction: {'see printed values above' }")
    print()

describe_stage(
    "num_leaves", stage1_results, ["num_leaves"],
    "A moderate value should win -- a very high leaf count risks fitting noise in the weakly "
    "correlated exogenous columns (EDA §7/§15) rather than real structure."
)
describe_stage(
    "max_depth", stage2_results, ["max_depth"],
    "Expected to matter less than num_leaves once num_leaves already caps complexity."
)
describe_stage(
    "learning_rate", stage3_results, ["learning_rate"],
    "A lower learning rate with more trees (capped by early stopping) should win or tie a higher "
    "learning rate with fewer trees, not lose to it."
)
describe_stage(
    "min_child_samples", stage4_results, ["min_child_samples"],
    "A higher value should help -- 340 series have under a year of history and 84 are >20% "
    "non-positive weeks (EDA §18), risking leaves that memorize a handful of unusual rows."
)
describe_stage(
    "colsample_bytree / subsample", stage5_results, ["colsample_bytree", "subsample"],
    "Some regularization value below 1.0 should help given the number of weak exogenous columns, "
    "without hiding Store/Dept/lag features from too many trees."
)
describe_stage(
    "reg_alpha / reg_lambda", stage6_results, ["reg_alpha", "reg_lambda"],
    "Expected to matter less than the structural settings above -- a secondary fine-tuning knob."
)
describe_stage(
    "objective", stage7_results, ["objective"],
    "WMAE is absolute-error-family -- regression_l1 or huber should align better with it than the "
    "squared-error default."
)
describe_stage(
    "use_sample_weight", stage8_results, ["use_sample_weight"],
    "Directly aligning the training loss with the 5x-holiday-weighted evaluation metric should help."
)
describe_stage(
    "use_exogenous", stage9_results, ["use_exogenous"],
    "Unlike Prophet, a tree model isn't forced to treat these as linear -- expected to help or be "
    "neutral, not hurt the way it helped Prophet to drop them."
)

--- num_leaves ---
Best:  {'num_leaves': np.int64(15)} -> WMAE=2233.79
Worst: {'num_leaves': np.int64(255)} -> WMAE=2382.76
Spread across this stage: 148.97 WMAE (material relative to best score)
Hypothesis was: A moderate value should win -- a very high leaf count risks fitting noise in the weakly correlated exogenous columns (EDA §7/§15) rather than real structure.
Consistent with hypothesis direction: see printed values above

--- max_depth ---
Best:  {'max_depth': np.int64(-1)} -> WMAE=2233.79
Worst: {'max_depth': np.int64(4)} -> WMAE=2248.62
Spread across this stage: 14.83 WMAE (small relative to best score)
Hypothesis was: Expected to matter less than num_leaves once num_leaves already caps complexity.
Consistent with hypothesis direction: see printed values above

--- learning_rate ---
Best:  {'learning_rate': np.float64(0.05)} -> WMAE=2199.39
Worst: {'learning_rate': np.float64(0.1)} -> WMAE=2229.94
Spread across this stage: 30.56 WMAE (small relative to best score)
Hypothesis 

In [20]:
best_use_exo = bool(stage9_results.loc[0, "use_exogenous"])
best_config = {**config_after_stage8, "use_exogenous": best_use_exo}
print("Final selected config:", best_config)

run = wandb.init(entity=WANDB_ENTITY, project=WANDB_PROJECT, group=WANDB_GROUP, name="LightGBM_Training_Final", job_type="training-full",
                  config={**best_config, "fold": "B_calendar_aligned"})

t0 = time.time()
final_wmae, final_trees = evaluate_config(best_config, cv_train, cv_val, verbose=True)
elapsed = time.time() - t0

wandb.log({"wmae": final_wmae, "seconds": elapsed, "n_trees": int(final_trees)})
print(f"Final CV WMAE: {final_wmae:.2f} (trees: {final_trees}) in {elapsed:.1f}s")
run.finish()

Final selected config: {'num_leaves': 15, 'max_depth': -1, 'learning_rate': np.float64(0.05), 'n_estimators': 3000, 'min_child_samples': 20, 'colsample_bytree': np.float64(0.6), 'subsample': np.float64(1.0), 'subsample_freq': 0, 'reg_alpha': np.float64(1.0), 'reg_lambda': np.float64(0.0), 'objective': 'regression', 'use_exogenous': True, 'use_sample_weight': False, 'use_christmas_shift': False}


wandb: setting up run 25xsjwpv


wandb: Tracking run with wandb version 0.28.0


wandb: Run data is saved locally in /Users/lizamarsagishvili/Downloads/final_project/wandb/run-20260709_215727-25xsjwpv
wandb: Run `wandb offline` to turn off syncing.


wandb: Syncing run LightGBM_Training_Final


...

wandb: uploading wandb-metadata.json; uploading requirements.txt; uploading wandb-summary.json; uploading config.yaml


wandb: uploading history steps 0-0, summary


wandb: 
wandb: Run history:
wandb: n_trees ▁
wandb: seconds ▁
wandb:    wmae ▁
wandb: 
wandb: Run summary:
wandb: n_trees 343
wandb: seconds 4.76535
wandb:    wmae 2113.37656
wandb: 


wandb: 🚀 View run LightGBM_Training_Final at: https://wandb.ai/toberi23-free-university-of-tbilisi-/Walmart-Recruiting-Store-Sales-Forecasting/runs/25xsjwpv
wandb: ⭐️ View project at: https://wandb.ai/toberi23-free-university-of-tbilisi-/Walmart-Recruiting-Store-Sales-Forecasting
wandb: Synced 4 W&B file(s), 0 media file(s), 0 artifact file(s) and 0 other file(s)


wandb: Find logs at: ./wandb/run-20260709_215727-25xsjwpv/logs


In [21]:
christmas_shift_config = {**best_config, "use_christmas_shift": True}

run = wandb.init(entity=WANDB_ENTITY, project=WANDB_PROJECT, group=WANDB_GROUP,
                  name="LightGBM_Training_Final_ChristmasShift", job_type="training-full",
                  config={**christmas_shift_config, "fold": "B_calendar_aligned"})

t0 = time.time()
final_wmae_shifted, shifted_trees = evaluate_config(christmas_shift_config, cv_train, cv_val, verbose=True)
elapsed = time.time() - t0

wandb.log({
    "wmae": final_wmae_shifted,
    "wmae_before_shift": final_wmae,
    "wmae_delta": final_wmae_shifted - final_wmae,
    "seconds": elapsed,
})
print(f"Final CV WMAE without shift: {final_wmae:.2f}")
print(f"Final CV WMAE with shift:    {final_wmae_shifted:.2f}  (expected to be ~equal -- CV's")
print("Christmas window is 2011, the adjuster only fires for 2012; see markdown above)")
run.finish()

best_config = christmas_shift_config
print("Final selected config (with Christmas shift):", best_config)

wandb: setting up run 0cmll13u


wandb: Tracking run with wandb version 0.28.0


wandb: Run data is saved locally in /Users/lizamarsagishvili/Downloads/final_project/wandb/run-20260709_215736-0cmll13u
wandb: Run `wandb offline` to turn off syncing.


wandb: Syncing run LightGBM_Training_Final_ChristmasShift


wandb: ⭐️ View project at https://wandb.ai/toberi23-free-university-of-tbilisi-/Walmart-Recruiting-Store-Sales-Forecasting


...

wandb: uploading history steps 0-0, summary


wandb: 
wandb: Run history:
wandb:           seconds ▁
wandb:              wmae ▁
wandb: wmae_before_shift ▁
wandb:        wmae_delta ▁
wandb: 
wandb: Run summary:
wandb:           seconds 2.25658
wandb:              wmae 2113.37656
wandb: wmae_before_shift 2113.37656
wandb:        wmae_delta 0
wandb: 


wandb: 🚀 View run LightGBM_Training_Final_ChristmasShift at: https://wandb.ai/toberi23-free-university-of-tbilisi-/Walmart-Recruiting-Store-Sales-Forecasting/runs/0cmll13u
wandb: ⭐️ View project at: https://wandb.ai/toberi23-free-university-of-tbilisi-/Walmart-Recruiting-Store-Sales-Forecasting
wandb: Synced 4 W&B file(s), 0 media file(s), 0 artifact file(s) and 0 other file(s)


wandb: Find logs at: ./wandb/run-20260709_215736-0cmll13u/logs


Final selected config (with Christmas shift): {'num_leaves': 15, 'max_depth': -1, 'learning_rate': np.float64(0.05), 'n_estimators': 3000, 'min_child_samples': 20, 'colsample_bytree': np.float64(0.6), 'subsample': np.float64(1.0), 'subsample_freq': 0, 'reg_alpha': np.float64(1.0), 'reg_lambda': np.float64(0.0), 'objective': 'regression', 'use_exogenous': True, 'use_sample_weight': False, 'use_christmas_shift': True}


In [22]:
final_pipeline = LGBMForecastPipeline(**best_config)
final_pipeline.fit(df_train)

with open("lightgbm_pipeline.pkl", "wb") as f:
    pickle.dump(final_pipeline, f)

run = wandb.init(entity=WANDB_ENTITY, project=WANDB_PROJECT, group=WANDB_GROUP, name="LightGBM_Pipeline_Save", job_type="save-pipeline",
                  config={**best_config, "cv_wmae_full": final_wmae})

artifact = wandb.Artifact("lightgbm_forecast_pipeline", type="model",
                           metadata={**best_config, "cv_wmae_full": final_wmae})
artifact.add_file("lightgbm_pipeline.pkl")
run.log_artifact(artifact)
wandb.log({"cv_wmae_full": final_wmae})
run.finish()

print("Saved lightgbm_pipeline.pkl and logged it as a W&B Artifact.")

wandb: setting up run xrocbv8g


wandb: Tracking run with wandb version 0.28.0


wandb: Run data is saved locally in /Users/lizamarsagishvili/Downloads/final_project/wandb/run-20260709_215749-xrocbv8g
wandb: Run `wandb offline` to turn off syncing.


wandb: Syncing run LightGBM_Pipeline_Save


wandb: ⭐️ View project at https://wandb.ai/toberi23-free-university-of-tbilisi-/Walmart-Recruiting-Store-Sales-Forecasting


...

wandb: uploading history steps 0-0, summary


wandb: 
wandb: Run history:
wandb: cv_wmae_full ▁
wandb: 
wandb: Run summary:
wandb: cv_wmae_full 2113.37656
wandb: 


wandb: 🚀 View run LightGBM_Pipeline_Save at: https://wandb.ai/toberi23-free-university-of-tbilisi-/Walmart-Recruiting-Store-Sales-Forecasting/runs/xrocbv8g
wandb: ⭐️ View project at: https://wandb.ai/toberi23-free-university-of-tbilisi-/Walmart-Recruiting-Store-Sales-Forecasting
wandb: Synced 4 W&B file(s), 0 media file(s), 2 artifact file(s) and 0 other file(s)


wandb: Find logs at: ./wandb/run-20260709_215749-xrocbv8g/logs


Saved lightgbm_pipeline.pkl and logged it as a W&B Artifact.
